In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
google_login_secret = os.getenv("google_login_secret")


In [3]:
from fastapi import FastAPI, Request, Depends
from fastapi.responses import RedirectResponse
from authlib.integrations.starlette_client import OAuth
from starlette.middleware.sessions import SessionMiddleware
from jose import jwt

import os

app = FastAPI()
app.add_middleware(SessionMiddleware, secret_key=google_login_secret)  # đổi secret

# Setup OAuth2
oauth = OAuth()
oauth.register(
    name='google',
    client_id=os.getenv("GOOGLE_CLIENT_ID"),
    client_secret=os.getenv("GOOGLE_CLIENT_SECRET"),
    server_metadata_url='https://accounts.google.com/.well-known/openid-configuration',
    client_kwargs={
        'scope': 'openid email profile'
    }
)

# Route để bắt đầu đăng nhập
@app.get("/login")
async def login(request: Request):
    redirect_uri = request.url_for('auth_callback')
    return await oauth.google.authorize_redirect(request, redirect_uri)

# Callback sau khi đăng nhập thành công
@app.get("/auth/callback")
async def auth_callback(request: Request):
    token = await oauth.google.authorize_access_token(request)
    user_info = await oauth.google.parse_id_token(request, token)

    email = user_info.get("email")
    name = user_info.get("name")
    student_id = email.split("@")[0]  # ví dụ: 2051234@dlu.edu.vn → 2051234

    # Tạo JWT token cho frontend
    jwt_token = jwt.encode(
        {"email": email, "student_id": student_id},
        "YOUR_SECRET_KEY",  # cần giống session key ở trên
        algorithm="HS256"
    )

    # Trả token về frontend (tạm thời redirect)
    response = RedirectResponse(url=f"/welcome?token={jwt_token}")
    return response

@app.get("/welcome")
async def welcome(token: str):
    return {"message": "Đăng nhập thành công!", "token": token}
